<a href="https://colab.research.google.com/github/huyle3/Data_eagles_wharton_comp/blob/main/DataEagles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
pure_dataset = pd.read_csv("Data/whl_2025.csv")

In [2]:
#same thing as general model

cols_to_drop = [ #Columns with object datatype besides gameid
    "went_ot", "home_off_line",
    "home_def_pairing", "away_off_line",
    "away_def_pairing", "home_goalie",
    "away_goalie", "record_id"
]
df_only_num = pure_dataset.drop(columns=cols_to_drop)



#reduce 10 records per game to 1; add from the game
df_agg = (
      df_only_num
      .groupby("game_id", as_index=False)
      .agg({
          "home_team": "first",
          "away_team": "first",
          "home_max_xg": "max",
          "away_max_xg": "max",
          **{
              col: "sum"
              for col in df_only_num.select_dtypes("number").columns
              if col != "id"
          }

      })
  )
#new column; home team wins = 1, away team wins = 0, 

df_agg["result"] = np.select(
    [
        df_agg["home_goals"] > df_agg["away_goals"],
        df_agg["home_goals"] <= df_agg["away_goals"],
    ],
    [1, 0]
)
df_agg.head()

,game_id,home_team,away_team,home_max_xg,away_max_xg,toi,home_assists,home_shots,home_xg,home_goals,away_assists,away_shots,away_xg,away_goals,home_penalties_committed,home_penalty_minutes,away_penalties_committed,away_penalty_minutes,result
0,game_1,thailand,pakistan,1.3905,1.2546,3599.99,2,21,2.8231,1,6,24,2.7516,3,8,16,6,12,0
1,game_10,switzerland,kazakhstan,1.2758,1.2719,3600.01,7,20,1.9254,4,4,30,3.3189,3,10,20,0,0,1
2,game_100,serbia,rwanda,1.7836,1.2909,3600.00,7,30,3.6712,4,10,27,3.0240,5,8,16,10,20,0
3,game_1000,brazil,netherlands,1.5239,1.1340,3600.03,6,32,3.5905,5,0,27,2.5261,0,10,20,6,12,1
4,game_1001,india,morocco,1.8355,1.2034,3599.99,4,32,3.4592,2,6,29,3.7658,3,8,16,8,16,0


## Create new per-game features
ex.
- xg_share

In [3]:
df_agg["home_xg_share"] = df_agg["home_xg"] / (df_agg["home_xg"] + df_agg["away_xg"])
df_agg["away_xg_share"] = df_agg["away_xg"] / (df_agg["away_xg"] + df_agg["home_xg"])

df_agg["home_shot_share"] = df_agg["home_shots"] / (df_agg["home_shots"] + df_agg["away_shots"])
df_agg["away_shot_share"] = df_agg["away_shots"] / (df_agg["home_shots"] + df_agg["away_shots"])

df_agg["home_xg_per_shot"] = df_agg["home_xg"] / df_agg["home_goals"]
df_agg["away_xg_per_shot"] = df_agg["away_xg"] / df_agg["away_goals"]

df_agg["home_goalie_goals_allowed_per_xg"] = df_agg["away_goals"] / df_agg["away_xg"]
df_agg["away_goalie_goals_allowed_per_xg"] = df_agg["home_goals"] / df_agg["home_xg"]








### Change to long format and change to averages   

In [4]:
home_cols = [c for c in df_agg.columns if c.startswith("home_") and c != "home_team"]
away_cols = [c for c in df_agg.columns if c.startswith("away_") and c != "away_team"]

home_df = df_agg[["game_id", "home_team", "away_team"] + home_cols].copy()

home_df = home_df.rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    **{c: c.replace("home_", "") for c in home_cols}
})

away_df = df_agg[["game_id", "away_team", "home_team"] + away_cols].copy()

away_df = away_df.rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    **{c: c.replace("away_", "") for c in away_cols}
})

team_games = pd.concat([home_df, away_df], ignore_index=True)

team_avgs = (
    team_games
    .groupby("team")
    .mean(numeric_only=True)
    .reset_index()
)

home_avg = team_avgs.add_prefix("home_avg_")

df_agg = df_agg.merge(
    home_avg,
    left_on="home_team",
    right_on="home_avg_team",
    how="left"
).drop(columns="home_avg_team")

away_avg = team_avgs.add_prefix("away_avg_")

df_agg = df_agg.merge(
    away_avg,
    left_on="away_team",
    right_on="away_avg_team",
    how="left"
).drop(columns="away_avg_team")



drop all the raw

In [5]:
df_agg = df_agg[[c for c in df_agg.columns if "avg" in c] + ["result"] + ["home_team"] + ["away_team"]]

create the diff features

In [ ]:
avg_cols = [c for c in df_agg.columns if c.startswith("home_avg_")]

for col in avg_cols:
    stat = col.replace("home_avg_", "")
    df_agg[f"diff_avg_{stat}"] = (
        df_agg[f"home_avg_{stat}"] - df_agg[f"away_avg_{stat}"]
    )

df_agg = df_agg[[c for c in df_agg.columns if c.startswith("diff_")] + ["result"] + ["home_team"] + ["away_team"]]


In [9]:
df_agg = df_agg.drop(columns=['diff_avg_xg_per_shot'])

In [10]:
df_agg.to_csv("df_1.csv", index=False)